In [71]:
import os
import pandas
from utils import partial_match_scores

dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
ds_name = "myriadlama-debug"
model_name = "qwen2.5_7b_it"

def get_filenames(
        modifyattn, modifyrope, scale_factor,
        repeat_paras, single_para_qapair, 
        num_paraphrases, num_samples=5):
    dump_file = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    if modifyattn:
        dump_file += "modifyattn."
    if modifyrope:
        dump_file += "modifyrope."
    if repeat_paras:
        dump_file += "repeatparas."
    if scale_factor > 0:
        dump_file += f"scalescore{str(int(scale_factor * 10))}."
    
    if single_para_qapair:
        dump_file += "singleparaqapair."


    dump_file += f"{num_samples}samples.{num_paraphrases}paras.feather"
    # print(f"Loading from {dump_file}")
    return dump_file

def _calculate_accuracy(df, label):
    predicts = [[pred] for pred in df["predict_lemma"].tolist()]
    answers = [answers for answers in df["answer_lemmas"]]
    acc = partial_match_scores(predicts, answers, birdirect=True)    
    print(f"{label} Accuracy: {acc:.4f}")

def calculate_accuracy(
        modifyattn, modifyrope, scale_factor,
        repeat_paras, single_para_qapair, 
        num_paraphrases, num_samples=5):
    filename = get_filenames(modifyattn, modifyrope, scale_factor, repeat_paras, single_para_qapair, num_paraphrases, num_samples=num_samples)
    if os.path.exists(filename) is False:
        print(f"File {filename} does not exist!")
        return None
    df = pandas.read_feather(filename)
    label = f"MyriadLlama {'+Attn' if modifyattn else ''} {'+Rope' if modifyrope else ''} {'+RepeatParas' if repeat_paras else ''} {num_paraphrases} Paras"
    _calculate_accuracy(df, label)
    return df

In [72]:
baseline_fn = f"{dataset_root}/{ds_name}/{model_name}/baseline_per_prompt.feather"
baseline = pandas.read_feather(baseline_fn)
baseline["predict_lemma"] = baseline["predict_lemma"].apply(lambda xs: xs[0])
_calculate_accuracy(baseline, "MyriadLlama Baseline Per Prompt")

MyriadLlama Baseline Per Prompt Accuracy: 0.4578


In [73]:
def report_accuracy(modifyattn, modifyrope, scale_factor, repeat_paras, single_para_qapair):
    para2 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_factor=scale_factor, repeat_paras=repeat_paras, single_para_qapair=single_para_qapair, num_paraphrases=2)
    para3 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_factor=scale_factor, repeat_paras=repeat_paras, single_para_qapair=single_para_qapair, num_paraphrases=3)
    para4 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_factor=scale_factor, repeat_paras=repeat_paras, single_para_qapair=single_para_qapair, num_paraphrases=4)
    para5 = calculate_accuracy(modifyattn=modifyattn, modifyrope=modifyrope, scale_factor=scale_factor, repeat_paras=repeat_paras, single_para_qapair=single_para_qapair, num_paraphrases=5)
    # return para2, para3, para4, para5

In [74]:
report_accuracy(True, False, scale_factor=0, repeat_paras=False, single_para_qapair=True)

File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.singleparaqapair.5samples.2paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.singleparaqapair.5samples.3paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.singleparaqapair.5samples.4paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.singleparaqapair.5samples.5paras.feather does not exist!


In [75]:
report_accuracy(True, True, scale_factor=0, repeat_paras=False, single_para_qapair=True)

File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.singleparaqapair.5samples.2paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.singleparaqapair.5samples.3paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.singleparaqapair.5samples.4paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.singleparaqapair.5samples.5paras.feather does not exist!


In [70]:
for scale_factor in [0.5, 1, 1.5, 2.0, 2.5, 3]:
    print(f"=== Scale Factor: {scale_factor} ===")
    report_accuracy(True, True, scale_factor=scale_factor, repeat_paras=False, single_para_qapair=True)
        

=== Scale Factor: 0.5 ===
MyriadLlama +Attn +Rope  2 Paras Accuracy: 0.3690
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.scalescore5.singleparaqapair.5samples.3paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.scalescore5.singleparaqapair.5samples.4paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.scalescore5.singleparaqapair.5samples.5paras.feather does not exist!
=== Scale Factor: 1 ===
MyriadLlama +Attn +Rope  2 Paras Accuracy: 0.3600
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modifyattn.modifyrope.scalescore10.singleparaqapair.5samples.3paras.feather does not exist!
File /home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama-debug/qwen2.5_7b_it/myriadlama.modif